In [1]:
from kafka import KafkaConsumer
import json
import matplotlib.pyplot as plt
import time

In [2]:
from collections import defaultdict

# buffers por sensor
# Buffers por sensor para inferencias
buffers = defaultdict(lambda: {
    "timestamps": [],
    "preds": [],
    "p0": [],
    "p1": [],
    # series filtradas por “clase escogida”
    "ts_pred1": [],  # timestamps cuando prediction==1
    "conf_pred1": [],# proba_class_1 cuando prediction==1
    "ts_pred0": [],  # timestamps cuando prediction==0
    "conf_pred0": [],# proba_class_0 cuando prediction==0
})

In [3]:


def initialize_consumer():
    kafka_topic = "water_quality_predict"
    kafka_bootstrap_servers = ["localhost:9092"]
    GROUP_ID = "water_quality_predict"
    return KafkaConsumer(
        kafka_topic,
        bootstrap_servers=kafka_bootstrap_servers,
        key_deserializer=lambda k: k.decode("utf-8") if k else "unknown_sensor",
        value_deserializer=lambda m: json.loads(m.decode("utf-8")),
        auto_offset_reset="latest",
        enable_auto_commit=True,
        group_id=GROUP_ID,
    )



In [4]:
def update_plot(consumer):
    try:
        for message in consumer:
            sensor_id = message.key if message.key else message.value.get("sensor_id", "unknown_sensor")
            sensor_data = message.value
            print(f"Received sensor={sensor_id}: {sensor_data}")

            ts = sensor_data["timestamp"]
            pred = int(sensor_data["prediction"])
            p0 = float(sensor_data["proba_class_0"])
            p1 = float(sensor_data["proba_class_1"])

            # buffer del sensor
            buf = buffers[sensor_id]

            # Update data storage
            buf["timestamps"].append(ts)
            buf["preds"].append(pred)
            buf["p0"].append(p0)
            buf["p1"].append(p1)

            # series filtradas por clase escogida
            if pred == 1:
                buf["ts_pred1"].append(ts)
                buf["conf_pred1"].append(p1)
            else:
                buf["ts_pred0"].append(ts)
                buf["conf_pred0"].append(p0)

            # Keep only the last 100 entries
            if len(buf["timestamps"]) > 100:
                min_ts = buf["timestamps"][-100]
                buf["timestamps"] = buf["timestamps"][-100:]
                buf["preds"] = buf["preds"][-100:]
                buf["p0"] = buf["p0"][-100:]
                buf["p1"] = buf["p1"][-100:]

                while buf["ts_pred1"] and buf["ts_pred1"][0] < min_ts:
                    buf["ts_pred1"].pop(0); buf["conf_pred1"].pop(0)
                while buf["ts_pred0"] and buf["ts_pred0"][0] < min_ts:
                    buf["ts_pred0"].pop(0); buf["conf_pred0"].pop(0)

            # Plot (2x2 con 3 paneles como la referencia)
            plt.figure(figsize=(10, 8))

            plt.subplot(2, 2, 1)
            plt.plot(buf["timestamps"], buf["preds"])
            plt.title("Aeration")
            plt.ylabel("Aeration produced")
            plt.ylim(-0.05, 1.05)

            plt.subplot(2, 2, 2)
            plt.plot(buf["ts_pred1"], buf["conf_pred1"], color="green")
            plt.title("Aeration confidence")
            plt.ylabel("Confidence")
            plt.ylim(0.0, 1.05)

            plt.subplot(2, 2, 3)
            plt.plot(buf["ts_pred0"], buf["conf_pred0"], color="orange")
            plt.title("Non aeration confidence")
            plt.ylabel("Confidence")
            plt.ylim(0.0, 1.05)

            plt.subplot(2, 2, 4)
            plt.axis("off")

            plt.tight_layout()
            plt.savefig(f"aeration_inference_{sensor_id}.png")
            plt.close()

            break  # Process one message at a time

    except KeyboardInterrupt:
        print("Stopped consuming messages.")
        consumer.close()

In [ ]:
consumer = initialize_consumer()
print("Subscribed to Kafka topic 'water_quality_predict'.")

try:
    while True:
        update_plot(consumer)
except KeyboardInterrupt:
    print("Stopped visualization.")
    consumer.close()

Subscribed to Kafka topic 'water_quality_predict'.
Received sensor=0: {'sensor_id': '0', 'timestamp': 1772370714, 'prediction': 0, 'proba_class_0': 1.0, 'proba_class_1': 0.0}
Received sensor=2: {'sensor_id': '2', 'timestamp': 1772370714, 'prediction': 1, 'proba_class_0': 0.0, 'proba_class_1': 1.0}
Received sensor=5: {'sensor_id': '5', 'timestamp': 1772370714, 'prediction': 1, 'proba_class_0': 0.1, 'proba_class_1': 0.9}
Received sensor=6: {'sensor_id': '6', 'timestamp': 1772370714, 'prediction': 1, 'proba_class_0': 0.1, 'proba_class_1': 0.9}
Received sensor=0: {'sensor_id': '0', 'timestamp': 1772370715, 'prediction': 1, 'proba_class_0': 0.3, 'proba_class_1': 0.7}
Received sensor=2: {'sensor_id': '2', 'timestamp': 1772370715, 'prediction': 0, 'proba_class_0': 1.0, 'proba_class_1': 0.0}
Received sensor=6: {'sensor_id': '6', 'timestamp': 1772370715, 'prediction': 0, 'proba_class_0': 1.0, 'proba_class_1': 0.0}
Received sensor=5: {'sensor_id': '5', 'timestamp': 1772370715, 'prediction': 1, '